# Praktické porovnanie implementácie na St. Anna Datasete
Tento malý prezentačný skript sa dá spustiť a uvidíte na ňom reálny (konkrétny) rozdiel medzi obmedzujúcim manuálnym zadávaním cez natívnu triedu AFwizard, a našim bleskovým autonomným ML riešením kódu v adresári tohto projektu .

In [ ]:
import afwizard as af
import json

# Načítanie skutočného LiDAR mračna (Vybrané zadanie: StA Lokalita)
# Pridáme af.add_filter_library, nech má afwizard vo vzorke natívne prístupy
af.add_filter_library("filters", recursive=False)

dataset = af.DataSet(filename="data/StA_last.laz", spatial_reference="EPSG:31256")
segmentation = af.load_segmentation("data/StA_segment.geojson", spatial_reference="EPSG:31256")

print("Dáta stiahnuté a nahraté do pamäte.")

### KROK 1: Prístup Ľudského Operátora (Originálne navhrnutý AFwizard pre UI)
Ak chceme spustiť túto pasáž, otvorí sa nám aktívne obrovské okno GUI. **Toto blokuje akékoľvek servery, paralelné cloud pipeline výpočty a dopytuje vizuálny systém používateľa.** Človek musí prepínať prepínače `step/offset` a skúšať Hillshade terén.

In [ ]:
# Ak odkomentujete tento príkaz, zastaví sa Python až kým manuálne človek nevyberie filtre:
print("Prebieha af.pipeline_tuning(dataset) -> vyvolanie interaktívneho prostredia do monitora operátora...")
# tuning = af.pipeline_tuning(dataset, segmentation_mapped=segmentation)
# tuning

---
### KROK 2: Návrh nášho Automatizovaného Systému Strojového Učenia
Naše riešenie toto UI úplne vynechalo. Extrahujeme si priamo na kóde geometriu priestoru, odošleme do modelu `RandomForest` a on vydá jasný príkaz, aký konkrétny konfiguračný súbor filtra nasadiť. Všetko bez interaktívnych prepínačov.

In [ ]:
import time
import sys
import os
sys.path.append(os.getcwd())

from feature_extraction import extract_segment_features
from ml_optimizer import MLFilterOptimizer

print("Spúšťam ML Model pre výpočet filtra...")
start_time = time.time()

# 1. Model na pozadí extrahuje hustotu bodov a ich drsnosť z mračna StA
features = extract_segment_features("data/StA_last.laz", "data/StA_segment.geojson")

# 2. Načítame natrénovaný ML Klasifikátor a predvídame iteráciu bez ľudského vizuálu
optimizer = MLFilterOptimizer()
optimizer.load_or_train_stub()

for seg_id, seg_geom in features.items():
    best_pipeline = optimizer.predict_best_filter(seg_geom)
    print(f"\n ---> [AUTONÓMNE ROZHODNUTIE MAPY '{seg_id}'] -> Stroj vybral ML filter: {best_pipeline} <--- ")
    
end_time = time.time()

print(f"\nCelkový čas neinteraktívneho rozhodovacieho stromu: {end_time - start_time:.4f} sekúnd.")
print("Tento výsledok by sa následne ihneď zapísal do geojson súboru (viď výstup nášho generovaného Docker obrazu).")